In [1]:
# ==========================================
# Cell 1 - Import Libraries
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================
# Cell 2 - Load Cleaned Dataset
# ==========================================

df = pd.read_csv("../data/processed/Nassau_Candy_Cleaned.csv")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Rows: 10194, Columns: 25


# 1. Route Definition & Aggregation

Each route is defined as **Factory → Customer State** (and separately **Factory → Customer Region**). For each route we compute: total shipments, average lead time, lead time variability (std dev), and delay frequency.

In [3]:
# ==========================================
# Cell 3 - Aggregate by Route State
# ==========================================

route_state_agg = df.groupby("Route State").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Lead Time (Days)", "mean"),
    Lead_Time_StdDev=("Lead Time (Days)", "std"),
    Delay_Frequency_Pct=("Delayed", lambda x: round(x.mean() * 100, 2)),
    Total_Sales=("Sales", "sum"),
    Total_Gross_Profit=("Gross Profit", "sum"),
).reset_index()

route_state_agg["Lead_Time_StdDev"] = route_state_agg["Lead_Time_StdDev"].fillna(0)
route_state_agg = route_state_agg.sort_values("Avg_Lead_Time")
route_state_agg.head(10)

,Route State,Total_Shipments,Avg_Lead_Time,Lead_Time_StdDev,Delay_Frequency_Pct,Total_Sales,Total_Gross_Profit
129,The Other Factory → North Carolina,1,0.000000,0.000000,0.0,9.00,7.00
63,Secret Factory → Connecticut,1,1.000000,0.000000,0.0,2.50,1.30
120,The Other Factory → Kentucky,2,1.000000,1.414214,0.0,26.00,2.00
54,Lot's O' Nuts → West Virginia,2,2.000000,2.828427,0.0,38.72,27.42
80,Secret Factory → New Mexico,2,2.000000,2.828427,0.0,12.50,6.50
84,Secret Factory → Oregon,2,2.000000,0.000000,0.0,180.00,90.00
76,Secret Factory → Nebraska,1,2.000000,0.000000,0.0,2.50,1.30
20,Lot's O' Nuts → Manitoba,2,2.000000,0.000000,0.0,18.00,12.00
125,The Other Factory → Nevada,1,2.000000,0.000000,0.0,16.25,1.25
117,The Other Factory → Florida,3,2.333333,1.527525,0.0,39.00,3.00


In [4]:
# ==========================================
# Cell 4 - Aggregate by Route Region
# ==========================================

route_region_agg = df.groupby("Route Region").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Lead Time (Days)", "mean"),
    Lead_Time_StdDev=("Lead Time (Days)", "std"),
    Delay_Frequency_Pct=("Delayed", lambda x: round(x.mean() * 100, 2)),
    Total_Sales=("Sales", "sum"),
    Total_Gross_Profit=("Gross Profit", "sum"),
).reset_index()

route_region_agg["Lead_Time_StdDev"] = route_region_agg["Lead_Time_StdDev"].fillna(0)
route_region_agg = route_region_agg.sort_values("Avg_Lead_Time")
route_region_agg

,Route Region,Total_Shipments,Avg_Lead_Time,Lead_Time_StdDev,Delay_Frequency_Pct,Total_Sales,Total_Gross_Profit
13,The Other Factory → Gulf,19,3.105263,1.969059,10.53,262.50,26.50
7,Secret Factory → Pacific,63,3.396825,1.879771,15.87,3076.25,1558.05
6,Secret Factory → Interior,45,3.577778,1.994183,20.00,1590.00,806.80
4,Secret Factory → Atlantic,72,3.847222,2.046362,25.00,2991.25,1499.85
15,The Other Factory → Pacific,32,3.906250,2.234038,28.12,386.25,61.25
10,Sugar Shack → Interior,8,4.000000,1.851640,12.50,44.74,23.39
0,Lot's O' Nuts → Atlantic,1661,4.014449,2.095844,29.32,22166.13,15315.63
12,The Other Factory → Atlantic,38,4.026316,2.098736,26.32,506.75,54.75
3,Lot's O' Nuts → Pacific,1813,4.029233,2.092171,28.74,24294.04,16788.44
5,Secret Factory → Gulf,37,4.135135,1.858347,27.03,930.00,480.00


# 2. Efficiency Benchmarking

Ranking routes fastest → slowest, and identifying the top 10 most efficient and bottom 10 least efficient Factory → State routes (minimum shipment volume applied so single-order routes don't dominate the ranking).

In [5]:
# ==========================================
# Cell 5 - Filter for Statistically Meaningful Routes
# ==========================================

MIN_SHIPMENTS = 10  # ignore routes with too few orders to be meaningful

meaningful_routes = route_state_agg[route_state_agg["Total_Shipments"] >= MIN_SHIPMENTS].copy()
print(f"{len(meaningful_routes)} of {len(route_state_agg)} routes have >= {MIN_SHIPMENTS} shipments")

96 of 196 routes have >= 10 shipments


In [6]:
# ==========================================
# Cell 6 - Top 10 Most Efficient Routes
# ==========================================

top_10_efficient = meaningful_routes.sort_values("Avg_Lead_Time").head(10)
top_10_efficient[["Route State", "Total_Shipments", "Avg_Lead_Time", "Delay_Frequency_Pct"]]

,Route State,Total_Shipments,Avg_Lead_Time,Delay_Frequency_Pct
157,Wicked Choccy's → Louisiana,15,3.066667,13.33
45,Lot's O' Nuts → Rhode Island,30,3.200000,16.67
90,Secret Factory → Texas,16,3.250000,6.25
184,Wicked Choccy's → Rhode Island,21,3.285714,14.29
139,Wicked Choccy's → Alabama,22,3.363636,13.64
67,Secret Factory → Illinois,13,3.384615,23.08
94,Secret Factory → Washington,11,3.454545,18.18
186,Wicked Choccy's → South Carolina,19,3.473684,15.79
167,Wicked Choccy's → Nebraska,14,3.500000,28.57
38,Lot's O' Nuts → Ohio,262,3.503817,23.66


In [7]:
# ==========================================
# Cell 7 - Bottom 10 Least Efficient Routes
# ==========================================

bottom_10_efficient = meaningful_routes.sort_values("Avg_Lead_Time", ascending=False).head(10)
bottom_10_efficient[["Route State", "Total_Shipments", "Avg_Lead_Time", "Delay_Frequency_Pct"]]

,Route State,Total_Shipments,Avg_Lead_Time,Delay_Frequency_Pct
179,Wicked Choccy's → Ontario,30,5.166667,40.00
4,Lot's O' Nuts → British Columbia,18,5.111111,44.44
33,Lot's O' Nuts → New Mexico,18,4.944444,44.44
183,Wicked Choccy's → Quebec,24,4.916667,29.17
159,Wicked Choccy's → Manitoba,10,4.900000,50.00
164,Wicked Choccy's → Mississippi,22,4.863636,45.45
170,Wicked Choccy's → New Hampshire,11,4.818182,36.36
188,Wicked Choccy's → Tennessee,67,4.776119,35.82
24,Lot's O' Nuts → Minnesota,43,4.720930,32.56
48,Lot's O' Nuts → Tennessee,109,4.660550,42.20


### ⚠️ Interpretation note

Because `Lead Time (Days)` is simulated from `Ship Mode` (see notebook 02), the routes at the top/bottom of this ranking mainly reflect **which ship modes are used on that route**, not independently observed delivery performance. Read this as *"routes that skew toward faster/slower shipping methods"* rather than proof of operational efficiency — call this out explicitly in the research paper.

# 3. Route Efficiency Score

A normalized 0-100 score per route, per the project KPI definition (normalized lead-time performance). Lower average lead time and lower delay frequency → higher score.

In [8]:
# ==========================================
# Cell 8 - Compute Route Efficiency Score
# ==========================================

def normalize_inverse(series):
    # lower raw value -> higher score (0-100)
    return 100 * (series.max() - series) / (series.max() - series.min())

meaningful_routes["Lead_Time_Score"] = normalize_inverse(meaningful_routes["Avg_Lead_Time"])
meaningful_routes["Delay_Score"] = normalize_inverse(meaningful_routes["Delay_Frequency_Pct"])

# Weighted: 60% lead time, 40% delay frequency
meaningful_routes["Route_Efficiency_Score"] = (
    0.6 * meaningful_routes["Lead_Time_Score"] + 0.4 * meaningful_routes["Delay_Score"]
).round(1)

efficiency_leaderboard = meaningful_routes.sort_values("Route_Efficiency_Score", ascending=False)
efficiency_leaderboard[["Route State", "Total_Shipments", "Avg_Lead_Time", "Delay_Frequency_Pct", "Route_Efficiency_Score"]].head(15)

,Route State,Total_Shipments,Avg_Lead_Time,Delay_Frequency_Pct,Route_Efficiency_Score
90,Secret Factory → Texas,16,3.250000,6.25,94.8
157,Wicked Choccy's → Louisiana,15,3.066667,13.33,93.5
45,Lot's O' Nuts → Rhode Island,30,3.200000,16.67,86.7
184,Wicked Choccy's → Rhode Island,21,3.285714,14.29,86.4
139,Wicked Choccy's → Alabama,22,3.363636,13.64,84.8
186,Wicked Choccy's → South Carolina,19,3.473684,15.79,79.6
94,Secret Factory → Washington,11,3.454545,18.18,78.0
12,Lot's O' Nuts → Idaho,14,3.857143,7.14,76.6
61,Secret Factory → California,31,3.580645,16.13,76.3
67,Secret Factory → Illinois,13,3.384615,23.08,75.5


# 4. Geographic Bottleneck Analysis

Regions/states with **high shipment volume AND poor performance** (below-median efficiency score) are the priority bottlenecks — they affect the most customers.

In [9]:
# ==========================================
# Cell 9 - Identify Bottleneck Routes
# ==========================================

median_score = meaningful_routes["Route_Efficiency_Score"].median()
median_volume = meaningful_routes["Total_Shipments"].median()

bottlenecks = meaningful_routes[
    (meaningful_routes["Route_Efficiency_Score"] < median_score) &
    (meaningful_routes["Total_Shipments"] >= median_volume)
].sort_values("Total_Shipments", ascending=False)

print(f"{len(bottlenecks)} high-volume, low-efficiency routes flagged as bottlenecks")
bottlenecks[["Route State", "Total_Shipments", "Avg_Lead_Time", "Route_Efficiency_Score"]].head(10)

29 high-volume, low-efficiency routes flagged as bottlenecks


,Route State,Total_Shipments,Avg_Lead_Time,Route_Efficiency_Score
144,Wicked Choccy's → California,823,4.228433,42.3
49,Lot's O' Nuts → Texas,569,4.228471,43.8
173,Wicked Choccy's → New York,436,4.178899,43.9
42,Lot's O' Nuts → Pennsylvania,331,4.314199,39.7
13,Lot's O' Nuts → Illinois,267,4.161049,47.4
181,Wicked Choccy's → Pennsylvania,227,4.224670,45.2
152,Wicked Choccy's → Illinois,210,4.176190,46.1
23,Lot's O' Nuts → Michigan,147,4.156463,44.7
36,Lot's O' Nuts → North Carolina,143,4.202797,41.9
2,Lot's O' Nuts → Arizona,111,4.099099,47.4


In [10]:
# ==========================================
# Cell 10 - Bottleneck by Region (state grouped up)
# ==========================================

region_bottleneck = route_region_agg.sort_values("Avg_Lead_Time", ascending=False)
region_bottleneck[["Route Region", "Total_Shipments", "Avg_Lead_Time", "Delay_Frequency_Pct"]]

,Route Region,Total_Shipments,Avg_Lead_Time,Delay_Frequency_Pct
9,Sugar Shack → Gulf,4,5.250000,50.00
8,Sugar Shack → Atlantic,18,4.722222,38.89
11,Sugar Shack → Pacific,3,4.666667,33.33
14,The Other Factory → Interior,11,4.272727,27.27
2,Lot's O' Nuts → Interior,1321,4.233157,31.64
16,Wicked Choccy's → Atlantic,1197,4.172097,30.33
17,Wicked Choccy's → Gulf,663,4.167421,29.71
19,Wicked Choccy's → Pacific,1342,4.157228,31.67
1,Lot's O' Nuts → Gulf,897,4.154961,31.44
18,Wicked Choccy's → Interior,950,4.140000,30.42


# 5. Ship Mode Performance Analysis

Comparing lead time, delay frequency, and cost-time tradeoffs across shipping methods.

In [11]:
# ==========================================
# Cell 11 - Ship Mode Comparison
# ==========================================

ship_mode_perf = df.groupby("Ship Mode").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Lead Time (Days)", "mean"),
    Delay_Frequency_Pct=("Delayed", lambda x: round(x.mean() * 100, 2)),
    Avg_Sales=("Sales", "mean"),
    Avg_Gross_Profit=("Gross Profit", "mean"),
).reset_index().sort_values("Avg_Lead_Time")

ship_mode_perf

,Ship Mode,Total_Shipments,Avg_Lead_Time,Delay_Frequency_Pct,Avg_Sales,Avg_Gross_Profit
1,Same Day,547,0.000000,0.00,13.004881,8.593656
0,First Class,1548,1.499354,0.00,13.772216,9.051092
2,Second Class,1979,2.984336,0.00,14.077928,9.250389
3,Standard Class,6120,5.502451,50.07,13.969011,9.219683


In [12]:
# ==========================================
# Cell 12 - Ship Mode Lead Time Boxplot
# ==========================================

fig = px.box(df, x="Ship Mode", y="Lead Time (Days)", color="Ship Mode",
             title="Lead Time Distribution by Ship Mode",
             category_orders={"Ship Mode": ["Same Day", "First Class", "Second Class", "Standard Class"]})
fig.show()

### Cost-time tradeoff (descriptive)

Faster ship modes don't inherently cost more in this dataset (Sales/Cost aren't tied to Ship Mode selection here), so a formal cost-time tradeoff can't be derived from this data as-is. What we *can* say descriptively: Standard Class carries both the longest lead times and the only meaningful delay risk, so it's the natural target for operational improvement.

# Observations

- 196 unique Factory → State routes and 20 Factory → Region routes were identified.
- After filtering for statistically meaningful volume (>=10 shipments), routes were ranked into a Route Efficiency Score (60% lead time, 40% delay frequency).
- Delay risk is concentrated entirely in Standard Class shipments (the only mode whose range crosses the 5-day threshold) — this is the highest-leverage lever for reducing delays.
- Geographic bottlenecks were flagged as high-volume + below-median-efficiency routes, prioritizing states that affect the most customers.
- **Caveat carried through from notebook 02**: since lead time is simulated from Ship Mode, route-level rankings should be read as "ship-mode mix by route," not independently verified courier performance. State this clearly in the paper's limitations section.

In [13]:
# ==========================================
# Cell 13 - Save Route Aggregates for Dashboard
# ==========================================

route_state_agg.to_csv("../data/processed/route_state_aggregates.csv", index=False)
route_region_agg.to_csv("../data/processed/route_region_aggregates.csv", index=False)
efficiency_leaderboard.to_csv("../data/processed/route_efficiency_leaderboard.csv", index=False)

print("Route aggregate files saved for Streamlit dashboard.")

Route aggregate files saved for Streamlit dashboard.
